# 08_FIX_2022_TIFF_ARTIFACTS

This notebook is a **patch for the existing repository workflow**. It does not retrain the models.

It loads the model bundles already saved by notebook 07 and regenerates the 2022 monthly and annual GeoTIFFs with corrected raster sampling.

### Fixes applied

1. **No `fill_nodata_nearest()`** — missing strips/cells remain missing.
2. **Pixel-center correction** is applied when converting lon/lat to array row/column indices.
3. Bilinear interpolation is used only where the local source neighborhood is valid.
4. No interpolation outside the real source extent.
5. Every prediction uses the intersection of valid pixels from all model features.
6. Outputs are written to a new folder, so old TIFFs are not overwritten.
7. Monthly and annual seam-QA tables are created.

In [ ]:
from pathlib import Path
from collections import OrderedDict
import re, math, warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import rasterio
from rasterio.transform import from_origin
from rasterio.features import geometry_mask
import geopandas as gpd
from scipy.ndimage import map_coordinates
import joblib

NODATA = -9999.0
TARGET_RES_DEG = 0.005
TEST_YEAR = 2022
SEAM_Z = 8.0

## 1. Locate repository and folders

In [ ]:
def find_project_root(start=None):
    cur = (Path(start) if start else Path.cwd()).resolve()
    for p in [cur, *cur.parents]:
        if (p/"data").exists() and (p/"notebooks").exists():
            return p
    raise FileNotFoundError("Repository root not found.")

PROJECT_ROOT = find_project_root()

DATA_DIR = PROJECT_ROOT/"data"
RAW_DIR = DATA_DIR/"raw"
PROCESSED_DIR = DATA_DIR/"processed"

ALIGNED_ROOT = PROCESSED_DIR/"aligned_rasters"
PRECIP_ROOT = ALIGNED_ROOT/"precipitation"
PRED_ROOT = ALIGNED_ROOT/"predictors"
RAW_PRED_ROOT = RAW_DIR/"predictors"

OLD_OUT = PROJECT_ROOT/"outputs"/"paper_style_proj_free_final"
MODEL_DIR = PROJECT_ROOT/"models"/"paper_style_proj_free_final"

OUT_ROOT = PROJECT_ROOT/"outputs"/"paper_style_fixed_no_artifacts"
MAP_DIR = OUT_ROOT/"monthly_masked_highres_tif"
ANNUAL_DIR = OUT_ROOT/"annual_masked_highres_tif"
QA_DIR = OUT_ROOT/"quality_control"

for p in [MAP_DIR, ANNUAL_DIR, QA_DIR]:
    p.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("MODEL_DIR:", MODEL_DIR)
print("NEW OUTPUT:", OUT_ROOT)

## 2. Rebuild the same dataset index used by notebook 07

In [ ]:
def norm_name(x):
    return re.sub(r"[^A-Z0-9]+","_",str(x).upper()).strip("_")

ALIASES = {
    "PERSIANN_CDR":"CDR",
    "PERSIANNCDR":"CDR",
    "PERSIANN_CCS":"CCS",
    "PDIR_NOW":"PDIR",
    "GSMAP_GAUGE_V7":"GSMAP_GAUGE",
    "GSMAP_GAUGE":"GSMAP_GAUGE",
    "GSMAP_MVK_V7":"GSMAP_MVK",
    "GSMAP_MVK":"GSMAP_MVK",
    "ERA5_TIFF":"ERA5",
    "CHIRPS_TIFF_2017_2022":"CHIRPS",
    "IMERG_MONTHLY":"IMERG",
    "LST_DAYTIME":"LST_DAY",
    "DISTANCE_TO_SEA":"DIST_SEA",
    "DISTANCE_SEA":"DIST_SEA",
    "ELEVATION":"DEM",
}

def canonical(x):
    n = norm_name(x)
    return ALIASES.get(n,n)

def all_tifs(folder):
    if folder is None or not Path(folder).exists():
        return []
    z=[]
    for pat in ["*.tif","*.tiff","*.TIF","*.TIFF"]:
        z.extend(Path(folder).rglob(pat))
    return sorted(set(z))

def folder_map(root):
    d={}
    if root.exists():
        for p in root.iterdir():
            if p.is_dir():
                d[canonical(p.name)] = p
    return d

def extract_ym(path):
    s=Path(path).stem
    m=re.search(r"(?<!\d)(20\d{2})[^0-9]+(0?[1-9]|1[0-2])(?!\d)",s)
    if m:
        return int(m.group(1)),int(m.group(2))
    m=re.search(r"(?<!\d)(20\d{2})(0[1-9]|1[0-2])(?!\d)",s)
    if m:
        return int(m.group(1)),int(m.group(2))
    return None

def month_index(folder):
    d={}
    for p in all_tifs(folder):
        ym=extract_ym(p)
        if ym is not None:
            d.setdefault(ym,p)
    return d

precip_folders = folder_map(PRECIP_ROOT)
predictor_folders = folder_map(PRED_ROOT)
raw_predictor_folders = folder_map(RAW_PRED_ROOT)

precip_index = {k:month_index(v) for k,v in precip_folders.items()}
predictor_index = {k:month_index(v) for k,v in predictor_folders.items()}

def monthly_path(name,year,month):
    key=(int(year),int(month))
    if name in precip_index:
        return precip_index[name].get(key)
    if name in predictor_index:
        return predictor_index[name].get(key)
    return None

print("Precip datasets:", sorted(precip_index))
print("Predictor datasets:", sorted(predictor_index))

## 3. Find static DEM and distance-to-sea rasters

In [ ]:
MONTHLY_LAND=[x for x in ["NDVI","LST_DAY"] if x in predictor_folders]

def choose_static_wgs84(name):
    candidates=[]
    for priority,roots in [(0,predictor_folders),(1,raw_predictor_folders)]:
        folder=roots.get(name)
        if not folder:
            continue
        for p in all_tifs(folder):
            if extract_ym(p) is not None:
                continue
            try:
                with rasterio.open(p) as src:
                    wkt=(src.crs.wkt if src.crs else "").upper()
                    is_geo=("GEOGCS" in wkt or "GEOGCRS" in wkt) and "WGS 84" in wkt
                    if is_geo:
                        candidates.append((priority,max(abs(src.res[0]),abs(src.res[1])),p))
            except Exception:
                pass
    if not candidates:
        return None
    return sorted(candidates,key=lambda z:(z[0],z[1]))[0][2]

static_land={}
for name in ["DEM","DIST_SEA"]:
    p=choose_static_wgs84(name)
    if p is not None:
        static_land[name]=p

def feature_path(feature,year,month):
    if feature in precip_folders or feature in MONTHLY_LAND:
        return monthly_path(feature,year,month)
    return static_land.get(feature)

print("Static land:", static_land)

## 4. Locate Khulna boundary and recreate the 0.005° output grid

In [ ]:
def find_boundary():
    candidates=[]
    for root in [RAW_DIR,PROCESSED_DIR,PROJECT_ROOT]:
        if not root.exists():
            continue
        for ext in ["*.shp","*.gpkg","*.geojson"]:
            for p in root.rglob(ext):
                s=p.name.lower()
                score=5*("khulna" in s)+3*("district" in s)+2*("bound" in s)
                candidates.append((score,p))
    if not candidates:
        raise FileNotFoundError("Khulna boundary file not found.")
    candidates.sort(key=lambda z:(-z[0],len(str(z[1]))))
    return candidates[0][1]

BOUNDARY_PATH=find_boundary()
boundary=gpd.read_file(BOUNDARY_PATH)
boundary=boundary[boundary.geometry.notna()].copy()

# Notebook 07 assumes geographic WGS84 for the final grid.
if boundary.crs is None:
    raise ValueError("Boundary CRS is missing.")

try:
    boundary=boundary.to_crs("EPSG:4326")
except Exception:
    # If local PROJ is broken but boundary already geographic WGS84, keep it.
    wkt=(boundary.crs.wkt if boundary.crs else "").upper()
    if not (("GEOGCS" in wkt or "GEOGCRS" in wkt) and "WGS 84" in wkt):
        raise

geom = boundary.geometry.union_all() if hasattr(boundary.geometry,"union_all") else boundary.unary_union
minx,miny,maxx,maxy = boundary.total_bounds

left=math.floor(minx/TARGET_RES_DEG)*TARGET_RES_DEG
right=math.ceil(maxx/TARGET_RES_DEG)*TARGET_RES_DEG
bottom=math.floor(miny/TARGET_RES_DEG)*TARGET_RES_DEG
top=math.ceil(maxy/TARGET_RES_DEG)*TARGET_RES_DEG

TARGET_WIDTH=int(round((right-left)/TARGET_RES_DEG))
TARGET_HEIGHT=int(round((top-bottom)/TARGET_RES_DEG))
TARGET_SHAPE=(TARGET_HEIGHT,TARGET_WIDTH)
TARGET_TRANSFORM=from_origin(left,top,TARGET_RES_DEG,TARGET_RES_DEG)

TARGET_MASK=geometry_mask(
    [geom.__geo_interface__],
    out_shape=TARGET_SHAPE,
    transform=TARGET_TRANSFORM,
    invert=True
)

TARGET_LONS=left+(np.arange(TARGET_WIDTH)+0.5)*TARGET_RES_DEG
TARGET_LATS=top-(np.arange(TARGET_HEIGHT)+0.5)*TARGET_RES_DEG
TG_LON,TG_LAT=np.meshgrid(TARGET_LONS,TARGET_LATS)

print("Boundary:",BOUNDARY_PATH)
print("Target shape:",TARGET_SHAPE)
print("Target transform:",TARGET_TRANSFORM)

## 5. Corrected raster sampler

This is the core fix.

The old workflow filled all internal NoData using the nearest valid cell and then sampled it. Here:

- NoData stays `NaN`.
- Geographic coordinates are converted to **array pixel-center coordinates** using `-0.5`.
- Bilinear interpolation is allowed only when the corresponding validity mask remains valid.

In [ ]:
def read_source(path):
    with rasterio.open(path) as src:
        a=src.read(1).astype("float32")
        if src.nodata is not None:
            a[np.isclose(a,src.nodata,equal_nan=True)] = np.nan
        a[np.abs(a)>1e30] = np.nan
        try:
            a[src.read_masks(1)==0] = np.nan
        except Exception:
            pass
        meta={
            "transform":src.transform,
            "height":src.height,
            "width":src.width,
            "bounds":src.bounds,
            "crs":src.crs
        }
    return a,meta

def sample_raster_fixed(path, order=1):
    if path is None or not Path(path).exists():
        return None

    a,meta=read_source(path)
    inv=~meta["transform"]

    # IMPORTANT:
    # inverse affine gives pixel-corner coordinates.
    # scipy array index 0 is the CENTER of first pixel.
    cols,rows=inv*(TG_LON,TG_LAT)
    cols=cols-0.5
    rows=rows-0.5

    inside=(
        (rows>=0) & (rows<=meta["height"]-1) &
        (cols>=0) & (cols<=meta["width"]-1)
    )

    out=np.full(TARGET_SHAPE,np.nan,dtype="float32")
    if not inside.any():
        return out

    coords=np.vstack([rows[inside],cols[inside]])

    # Interpolate values.
    # NaN is temporarily set to zero, BUT a separate validity interpolation
    # makes sure any neighborhood touched by missing data is rejected.
    valid_src=np.isfinite(a).astype("float32")
    value_src=np.where(np.isfinite(a),a,0).astype("float32")

    sampled_val=map_coordinates(
        value_src,coords,order=order,mode="constant",cval=0.0
    )
    sampled_valid=map_coordinates(
        valid_src,coords,order=order,mode="constant",cval=0.0
    )

    # For bilinear order=1, require virtually complete valid support.
    good=sampled_valid >= 0.999

    tmp=np.full(sampled_val.shape,np.nan,dtype="float32")
    tmp[good]=sampled_val[good].astype("float32")
    out[inside]=tmp
    out[~TARGET_MASK]=np.nan
    return out

print("Corrected sampler ready.")

## 6. QA: compare old and corrected sampling for one raster

In [ ]:
example=None
for name in ["CHIRPS","CDR","ERA5","PERSIANN","NDVI","LST_DAY"]:
    p=feature_path(name,TEST_YEAR,1)
    if p is not None:
        example=(name,p)
        break

if example:
    name,p=example
    fixed=sample_raster_fixed(p)
    print("Example:",name,p)
    print("Fixed valid target pixels:",np.isfinite(fixed[TARGET_MASK]).sum(),"/",TARGET_MASK.sum())
    print("Fixed range:",np.nanmin(fixed),np.nanmax(fixed))

## 7. Load saved model bundles from notebook 07

In [ ]:
model_files=sorted(MODEL_DIR.glob("*.joblib"))
if not model_files:
    raise FileNotFoundError(
        f"No saved models found in {MODEL_DIR}. Run notebook 07 through the model-training cell first."
    )

bundles={}
for p in model_files:
    try:
        b=joblib.load(p)
        if not isinstance(b,dict) or "estimator" not in b or "features" not in b:
            continue
        stem=p.stem
        if "__" not in stem:
            continue
        combo,model=stem.split("__",1)
        model=model.replace("_"," ")
        bundles[(combo,model)] = b
    except Exception as e:
        print("Could not load:",p.name,e)

print("Usable model bundles:",len(bundles))
for k,b in list(bundles.items())[:20]:
    print(k,"features=",b["features"])

## 8. Fixed monthly prediction function

In [ ]:
def predict_month_fixed(bundle,year,month):
    arrays=[]
    missing=[]

    for f in bundle["features"]:
        p=feature_path(f,year,month)
        if p is None or not Path(p).exists():
            missing.append(f)
            continue

        a=sample_raster_fixed(p,order=1)
        if a is None:
            missing.append(f)
            continue

        if f=="NDVI":
            finite=a[np.isfinite(a)]
            if finite.size and np.nanpercentile(np.abs(finite),95)>2:
                a=a*0.0001

        if f in precip_folders:
            a[a<0]=np.nan

        arrays.append((f,a))

    if missing:
        return None,"missing file: "+", ".join(missing)

    X=np.column_stack([a.ravel() for _,a in arrays])
    valid=np.all(np.isfinite(X),axis=1) & TARGET_MASK.ravel()

    if valid.sum()==0:
        return None,"no common valid target pixels"

    pred=np.full(TARGET_SHAPE,np.nan,dtype="float32")
    pred.ravel()[valid]=np.maximum(
        bundle["estimator"].predict(X[valid]),0
    ).astype("float32")
    pred[~TARGET_MASK]=np.nan
    return pred,"OK"

## 9. GeoTIFF writer

In [ ]:
def reference_crs():
    for name in ["CHIRPS","CDR","ERA5","PERSIANN"]:
        p=feature_path(name,TEST_YEAR,1)
        if p:
            with rasterio.open(p) as src:
                return src.crs
    raise RuntimeError("No reference CRS found.")

REF_CRS=reference_crs()

def write_tif(path,arr):
    path=Path(path)
    path.parent.mkdir(parents=True,exist_ok=True)

    out=np.where(np.isfinite(arr)&TARGET_MASK,arr,NODATA).astype("float32")

    profile={
        "driver":"GTiff",
        "height":TARGET_HEIGHT,
        "width":TARGET_WIDTH,
        "count":1,
        "dtype":"float32",
        "crs":REF_CRS,
        "transform":TARGET_TRANSFORM,
        "nodata":NODATA,
        "compress":"LZW",
        "tiled":True
    }

    with rasterio.open(path,"w",**profile) as dst:
        dst.write(out,1)

## 10. Seam diagnostic

In [ ]:
def robust_seam(arr):
    row=np.nanmean(arr,axis=1)
    col=np.nanmean(arr,axis=0)

    def calc(v):
        d=np.abs(np.diff(v))
        f=d[np.isfinite(d)]
        if len(f)<2:
            return np.nan,-1
        med=np.nanmedian(f)
        mad=np.nanmedian(np.abs(f-med))
        scale=max(1e-12,1.4826*mad)
        z=(d-med)/scale
        if not np.isfinite(z).any():
            return np.nan,-1
        i=int(np.nanargmax(z))
        return float(z[i]),i

    rz,ri=calc(row)
    cz,ci=calc(col)
    return rz,ri,cz,ci

## 11. Regenerate all 2022 monthly maps

This uses **all saved model bundles** found in the model directory, not only a hard-coded subset.

In [ ]:
monthly_registry={}
qa=[]

for (combo,model),bundle in sorted(bundles.items()):
    print("\\nPredicting:",combo,"|",model)
    outpaths=[]

    for month in range(1,13):
        pred,status=predict_month_fixed(bundle,TEST_YEAR,month)

        if pred is None:
            qa.append({
                "combo":combo,"model":model,"month":month,
                "status":"SKIP","reason":status
            })
            print(month,"SKIP",status)
            continue

        out=MAP_DIR/combo/model.replace(" ","_")/f"downscaled_{TEST_YEAR}_{month:02d}.tif"
        write_tif(out,pred)
        outpaths.append(out)

        rz,ri,cz,ci=robust_seam(pred)
        qa.append({
            "combo":combo,"model":model,"month":month,
            "status":"OK","reason":"",
            "valid_pixels":int(np.isfinite(pred[TARGET_MASK]).sum()),
            "valid_pct":float(np.isfinite(pred[TARGET_MASK]).mean()*100),
            "min":float(np.nanmin(pred)),
            "max":float(np.nanmax(pred)),
            "mean":float(np.nanmean(pred)),
            "max_row_seam_z":rz,"row_index":ri,
            "max_col_seam_z":cz,"col_index":ci
        })

        print(f"{month:02d} OK | {np.nanmin(pred):.1f}-{np.nanmax(pred):.1f}")

    monthly_registry[(combo,model)] = outpaths

qa_df=pd.DataFrame(qa)
qa_df.to_csv(QA_DIR/"map_generation_FIXED_QA.csv",index=False)
display(qa_df)

## 12. Generate annual maps from the 12 corrected monthly maps

In [ ]:
annual_qa=[]

for (combo,model),paths in monthly_registry.items():
    if len(paths)!=12:
        annual_qa.append({
            "combo":combo,"model":model,"status":"SKIP",
            "reason":f"{len(paths)}/12 months"
        })
        continue

    arrs=[]
    for p in paths:
        with rasterio.open(p) as src:
            a=src.read(1).astype("float32")
            if src.nodata is not None:
                a[a==src.nodata]=np.nan
            arrs.append(a)

    stack=np.stack(arrs,axis=0)
    valid=np.all(np.isfinite(stack),axis=0)&TARGET_MASK

    annual=np.full(TARGET_SHAPE,np.nan,dtype="float32")
    annual[valid]=np.sum(stack[:,valid],axis=0)

    out=ANNUAL_DIR/f"annual_{TEST_YEAR}_{combo}_{model.replace(' ','_')}_FIXED.tif"
    write_tif(out,annual)

    rz,ri,cz,ci=robust_seam(annual)
    annual_qa.append({
        "combo":combo,"model":model,"status":"OK","reason":"",
        "valid_pct":float(np.isfinite(annual[TARGET_MASK]).mean()*100),
        "min":float(np.nanmin(annual)),
        "max":float(np.nanmax(annual)),
        "mean":float(np.nanmean(annual)),
        "max_row_seam_z":rz,"row_index":ri,
        "max_col_seam_z":cz,"col_index":ci,
        "seam_flag":bool((np.isfinite(rz) and rz>=SEAM_Z) or (np.isfinite(cz) and cz>=SEAM_Z)),
        "path":str(out)
    })

annual_df=pd.DataFrame(annual_qa)
annual_df.to_csv(QA_DIR/"annual_generation_FIXED_QA.csv",index=False)
display(annual_df)

## 13. Final diagnosis summary

In [ ]:
ok_month=qa_df[qa_df.status=="OK"] if len(qa_df) else pd.DataFrame()
flag_month=ok_month[
    (ok_month["max_row_seam_z"].fillna(-999)>=SEAM_Z) |
    (ok_month["max_col_seam_z"].fillna(-999)>=SEAM_Z)
] if len(ok_month) else pd.DataFrame()

flag_annual=annual_df[
    annual_df.get("seam_flag",False)==True
] if len(annual_df) and "seam_flag" in annual_df else pd.DataFrame()

lines=[
    "2022 TIFF ARTIFACT FIX REPORT",
    "="*70,
    f"Monthly maps OK: {len(ok_month)}",
    f"Monthly seam flags: {len(flag_month)}",
    f"Annual maps generated: {int((annual_df.status=='OK').sum()) if len(annual_df) else 0}",
    f"Annual seam flags: {len(flag_annual)}",
    "",
    "FIXES:",
    "1. Removed global nearest-neighbour filling of internal NoData.",
    "2. Applied -0.5 pixel-center correction before scipy map_coordinates.",
    "3. Interpolation rejected if source support includes NoData.",
    "4. Prediction restricted to common valid predictor pixels.",
    "5. Annual output requires 12 valid monthly maps.",
    "",
    f"Outputs: {OUT_ROOT}"
]

report=OUT_ROOT/"FIX_REPORT.txt"
report.write_text("\\n".join(lines),encoding="utf-8")
print(report.read_text(encoding="utf-8"))

### Send back after running

Please send:

- `paper_style_fixed_no_artifacts/FIX_REPORT.txt`
- `quality_control/map_generation_FIXED_QA.csv`
- `quality_control/annual_generation_FIXED_QA.csv`

Then the remaining problem, if any, can be traced to the specific source predictor/month.